# Brachistochrone solver — usageMinimum-**time** descent on a smooth, frictionless track, solved by Dijkstra overa discretised state space. See `README.md` for the method.> The cost minimised here is **time, not distance**. The cycloid is *longer* than> the straight chord and still arrives sooner — that is the whole point of the> problem. Path length is reported alongside but never optimised.**Contents**1. Basic usage — solve, inspect the array, sample the path2. 2D curves across a range of heights3. 2D curves across a range of θ/π, including values above 14. 3D state space for the 100 m → 0 drop with a sub-target dip

In [ ]:
import mathimport matplotlib.pyplot as pltimport numpy as npfrom brachistochrone import (    Config,    solve,    analytic_brachistochrone,    cycloid_curve,    endpoint_for_theta_ratio,)%matplotlib inlineplt.rcParams["figure.figsize"] = (10, 5)plt.rcParams["figure.dpi"] = 110

## 1. Basic usageFour calls cover the whole API.

In [ ]:
sol = solve(Config(h=100.0, y_end=0.0, dt=0.1))arr = sol.as_structured_array()          # (32, 21, 10)path = sol.path_nodes(sol.target_index(100.0), sol.iy_target)samples = sol.sample_path(path)          # t, x, y, v, angle every dtprint(f"grid            {sol.shape}")print(f"x_max           {sol.x_max:.3f} m  (limited by {sol.x_max_binding})")print(f"reachable       {int(np.isfinite(sol.time).sum())} of {arr.size} states")print(f"path to x=100   {len(path)} nodes, {len(samples)} samples at dt=0.1 s")

### A note on `iy_target`The snippet in the docs uses `sol.shape[1] - 1` for the target row. That iscorrect *only* when the grid has no headroom below the target altitude — which isthe default. Once `theta_ratio > 1` or `depth_below` is set, the grid extendsbelow `y_end` and the last row is no longer the target.`sol.iy_target` is always right, so prefer it.

In [ ]:
plain = solve(Config())dipped = solve(Config(theta_ratio=1.29))for name, s in (("default", plain), ("theta_ratio=1.29", dipped)):    print(f"{name:18s} shape={s.shape}  iy_target={s.iy_target:2d}  "          f"last row={s.shape[1] - 1:2d}  y[iy_target]={s.y[s.iy_target]:5.1f} m  "          f"grid floor={s.y[-1]:6.1f} m")

### The state-space array`as_structured_array()` returns one record per `(ix, iy, ir)` state. `t_min` and`path_len` describe the best path *to* that state; `dx`, `dy`, `v_in` and`angle_in_deg` describe the final step that got there.

In [ ]:
print("fields:", ", ".join(arr.dtype.names))print()# The far bottom corner, at whichever heading reaches it fastest.ix = sol.shape[0] - 1t_best, ir_best = sol.best_at(ix, sol.iy_target)cell = arr[ix, sol.iy_target, ir_best]for field in arr.dtype.names:    print(f"  {field:14s} {cell[field]}")

Because speed depends only on altitude, `v` is constant across every row — auseful sanity check on the whole method.

In [ ]:
v_by_row = arr["v"][:, :, 0]print("max spread of v within an altitude row:", np.ptp(v_by_row, axis=0).max())print("matches sqrt(2*g*(h-y))?         ",      np.allclose(arr["v"][0, :, 0], np.sqrt(2 * 9.81 * (100.0 - sol.y))))

### The sampled path`sample_path` resamples the winning polyline onto a uniform `dt` grid. Withineach chord the motion is uniformly accelerated, so the interpolation is exact.

In [ ]:
print(f"{'t':>6} {'x':>8} {'y':>8} {'v':>7} {'angle':>7}")for row in samples[::8]:    print(f"{row['t']:6.2f} {row['x']:8.2f} {row['y']:8.2f} "          f"{row['v']:7.2f} {row['angle_deg']:7.1f}")print(f"{'...':>6}")last = samples[-1]print(f"{last['t']:6.2f} {last['x']:8.2f} {last['y']:8.2f} "      f"{last['v']:7.2f} {last['angle_deg']:7.1f}")

The samples bunch near the top and spread along the floor — the ball spends adisproportionate share of the run at low speed. That is exactly why the optimalcurve dives steeply at the start.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))ax.plot(*zip(*path), color="0.7", lw=1.2, zorder=1, label="chord polyline")sc = ax.scatter(samples["x"], samples["y"], c=samples["t"], cmap="viridis",                s=26, zorder=2)fig.colorbar(sc, ax=ax, label="t (s)")ax.set_xlabel("x (m)"); ax.set_ylabel("altitude (m)")ax.set_title(f"Ball position every dt = 0.1 s  ({len(samples)} samples)")ax.set_aspect("equal"); ax.grid(True, color="0.92"); ax.legend()plt.show()

## 2. A range of heightsEach solve uses its own default extent, `x_max = π·drop/2`.

In [ ]:
heights = [25.0, 50.0, 100.0, 200.0, 400.0]solutions = {}fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))colours = plt.get_cmap("viridis")(np.linspace(0.1, 0.85, len(heights)))for h, colour in zip(heights, colours):    s = solve(Config(h=h, y_end=0.0, x_step=h / 20, y_step=h / 20))    solutions[h] = s    seg = s.path_segments(s.shape[0] - 1, s.iy_target)    t, _ = s.best_at(s.shape[0] - 1, s.iy_target)    xs = np.append(seg["x1"], seg["x2"][-1])    ys = np.append(seg["y1"], seg["y2"][-1])    ax1.plot(xs, ys, color=colour, lw=2, label=f"h = {h:g} m   t = {t:.3f} s")    ax2.plot(xs / h, ys / h, color=colour, lw=2, label=f"h = {h:g} m")ax1.set_xlabel("x (m)"); ax1.set_ylabel("altitude (m)")ax1.set_title("Physical units"); ax1.set_aspect("equal")ax2.set_xlabel("x / h"); ax2.set_ylabel("altitude / h")ax2.set_title("Normalised by height — the curves collapse")ax2.set_aspect("equal")for a in (ax1, ax2):    a.grid(True, color="0.92"); a.legend(fontsize=8)plt.tight_layout(); plt.show()

The right panel is the point: **the shape is scale-invariant.** Normalising byheight collapses every curve onto one, because the cycloid is fixed by the*ratio* `x_end / drop`, and each solve uses the same ratio (`π/2`). Height setsthe clock, not the geometry.Descent time scales as `√h`:

In [ ]:
print(f"{'h (m)':>8} {'t (s)':>8} {'t/sqrt(h)':>11} {'length (m)':>11} {'len/h':>7}")for h, s in solutions.items():    t, _ = s.best_at(s.shape[0] - 1, s.iy_target)    seg = s.path_segments(s.shape[0] - 1, s.iy_target)    L = seg["length"].sum()    print(f"{h:8.0f} {t:8.4f} {t / math.sqrt(h):11.5f} {L:11.2f} {L / h:7.4f}")

## 3. A range of θ/π, including values above 1`theta_ratio` sets the endpoint by the cycloid parameter: `θ = ratio·π`. Thecycloid's lowest point is always at `θ = π`, so:- **ratio ≤ 1** — the endpoint *is* the lowest point, path descends monotonically- **ratio > 1** — the path dips *below* the target and climbs backPast the threshold the solver needs grid headroom below `y_end` and upward moves;both are enabled automatically when `theta_ratio > 1`.

In [ ]:
ratios = [0.5, 0.75, 1.0, 1.117, 1.216, 1.29]print(f"{'θ/π':>6} {'x_end (m)':>10} {'dip (m)':>9} {'climbs':>7} {'t (s)':>8} {'err':>7}")fig, ax = plt.subplots(figsize=(12, 5.5))colours = plt.get_cmap("plasma")(np.linspace(0.05, 0.8, len(ratios)))for ratio, colour in zip(ratios, colours):    s = solve(Config(theta_ratio=ratio, x_step=2.5, y_step=2.5))    ixt = s.shape[0] - 1    seg = s.path_segments(ixt, s.iy_target)    t, _ = s.best_at(ixt, s.iy_target)    x_end, dip_ref = endpoint_for_theta_ratio(ratio, 100.0)    t_ref, _ = analytic_brachistochrone(x_end, 100.0, 9.81)    xs = np.append(seg["x1"], seg["x2"][-1])    ys = np.append(seg["y1"], seg["y2"][-1])    ax.plot(xs, ys, color=colour, lw=2.2, label=f"θ/π = {ratio:.3f}")    if dip_ref > 0:        k = int(np.argmin(ys))        ax.plot(xs[k], ys[k], marker="v", color=colour, ms=10, mec="k", mew=0.8)    print(f"{ratio:6.3f} {x_end:10.2f} {dip_ref:9.2f} "          f"{int((seg['dy'] > 1e-9).sum()):7d} {t:8.4f} {(t - t_ref) / t_ref:6.2%}")ax.axhline(0, color="k", lw=1.2, ls="--")ax.text(0.99, 0.03, "target altitude", transform=ax.transAxes,        ha="right", fontsize=9)ax.set_xlabel("x (m)"); ax.set_ylabel("altitude (m)")ax.set_title("Descent from 100 m for a range of θ/π   "             "(▼ = lowest point, falls before the endpoint)")ax.set_aspect("equal"); ax.grid(True, color="0.92"); ax.legend(fontsize=9)plt.show()

Note the `climbs` column: zero until θ/π passes 1, then non-zero. That is thesolver genuinely using upward chords, not an artefact of plotting.The dip is a **loan repaid in full** — the track is frictionless, so all thekinetic energy gained on the way down is returned on the way up. Speed overshootsthe free-fall value and comes back to it exactly:

In [ ]:
s = solve(Config(theta_ratio=1.29, x_step=2.5, y_step=2.5))seg = s.path_segments(s.shape[0] - 1, s.iy_target)t_nodes = np.append(seg["t_start"], seg["t_end"][-1])v_nodes = np.append(seg["v1"], seg["v2"][-1])v_ref = math.sqrt(2 * 9.81 * 100.0)fig, ax = plt.subplots(figsize=(10, 4))ax.plot(t_nodes, v_nodes, lw=2, color="#1f77b4", label="speed along the path")ax.axhline(v_ref, color="0.5", ls="--", label=f"sqrt(2g·drop) = {v_ref:.2f} m/s")ax.set_xlabel("t (s)"); ax.set_ylabel("speed (m/s)")ax.set_title("θ/π = 1.29 — speed overshoots in the dip, returns exactly on arrival")ax.grid(True, color="0.92"); ax.legend()plt.show()print(f"peak speed    {v_nodes.max():.3f} m/s")print(f"arrival speed {v_nodes[-1]:.3f} m/s   (free-fall value {v_ref:.3f})")

## 4. 3D state space, 100 m → 0, with θ/π > 1Every reachable `(ix, iy, ir)` state, coloured by the path length reaching it.The plot helper from `plot_states` does the work.With `theta_ratio > 1` the heading axis becomes **signed** (−90°…90°, 19 bins) soascending chords carry honest negative angles, and the grid extends below thetarget — marked by the dashed rectangle.

In [ ]:
from plot_states import plotsol3d = solve(Config(theta_ratio=1.29))fig, ax = plot(sol3d, interactive=False, elev=20, azim=-62)fig.set_size_inches(12, 8.5)plt.show()print(f"heading axis : {sol3d.r_deg[0]:+.0f}° … {sol3d.r_deg[-1]:+.0f}° "      f"in {sol3d.shape[2]} bins")print(f"grid floor   : {sol3d.y[-1]:.2f} m "      f"({sol3d.depth_below:.2f} m below the target)")

Outside a notebook, run it interactively for rotation on all three axes —mouse drag gives azimuth and elevation, and sliders cover roll as well:```bashpython plot_states.py --theta-ratio 1.29```Passing `interactive=False` above suppresses those sliders, which need a livebackend to be useful.

### Comparing several θ/π in one 3D viewOverlaying the optimal paths alone, without the point clouds, shows how thefamily sweeps below the target as θ/π grows.

In [ ]:
fig = plt.figure(figsize=(12, 8))ax = fig.add_subplot(111, projection="3d")colours = plt.get_cmap("plasma")(np.linspace(0.05, 0.8, len(ratios)))for ratio, colour in zip(ratios, colours):    s = solve(Config(theta_ratio=ratio, x_step=2.5, y_step=2.5))    ixt = s.shape[0] - 1    _, ir = s.best_at(ixt, s.iy_target)    states = s.path_states(ixt, s.iy_target, ir)    lx = np.array([s.x[j] for j, _, _ in states])    lr = np.array([s.r_deg[r] for _, _, r in states])    lz = np.array([s.y[i] for _, i, _ in states])    lr[0] = lr[1]                      # start is at rest: no meaningful heading    ax.plot(lx, lr, lz, color=colour, lw=2.4, label=f"θ/π = {ratio:.3f}")ax.set_xlabel("x (m)", labelpad=10)ax.set_ylabel("heading r (deg)", labelpad=10)ax.set_zlabel("altitude (m)", labelpad=8)ax.set_title("Optimal paths through state space for a range of θ/π")ax.view_init(elev=22, azim=-62)ax.legend(fontsize=9, loc="upper left")plt.show()

## Gotchas- **`sol.iy_target`, not `sol.shape[1] - 1`.** They coincide only when the grid  has no sub-target headroom.- **Grid steps are snapped**, not used verbatim, so the axes span their range  exactly. Requesting `y_step=3.3` over a 100 m drop realises as 3.3333 m.- **`theta_ratio` and `x_max` are mutually exclusive** — passing both raises  `TypeError`.- **`theta_ratio` must be in `(0, 2)`.** As it approaches 2 the arch flattens and  `x_end` diverges.- **From rest, a horizontal first move is impossible** and is correctly reported  as unreachable (infinite time) rather than raising.- **Results are upper bounds.** Every grid polyline is an admissible path, so the  discrete time can never beat the continuous optimum. Refine the grid to tighten  it — the error roughly halves per refinement.